In [1]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

In [2]:
columns = [ "age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation", 
           "relationship", "race", "sex", "capital_gain", "capital_loss", "hours_per_week", "native_country", "income" ] 

In [4]:
df_train = pd.read_csv(r"C:\Users\gowri\OneDrive\adult.data", header=None, names=columns, na_values="?", skipinitialspace=True) 

In [5]:
df_train.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [10]:
df_test = pd.read_csv(r"C:\Users\gowri\OneDrive\adult\adult.test", header=None, names=columns, na_values="?", skipinitialspace=True, skiprows=1)

In [11]:
df_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K.


In [12]:
df_train["source"] = "train"
df_test["source"] = "test"

In [13]:
df_train_test = pd.concat([df_train ,df_test], ignore_index=True)

In [14]:
df_train_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train


In [15]:
df_train_test.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [16]:
df_train_test.isnull().sum()

age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
source               0
dtype: int64

In [17]:
df_train_test['sex'].unique()

array(['Male', 'Female'], dtype=object)

In [18]:
df_train_test['income'].unique()

array(['<=50K', '>50K', '<=50K.', '>50K.'], dtype=object)

In [19]:
df_train_test['income'].value_counts()

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [20]:
df_train_test['income'] = df_train_test['income'].str.replace('.', '', regex=False).str.strip()

In [21]:
df_train_test['income'].value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

<h3 style="color:#1A5276;">Hypothesis Question 1: Does gender affect the probability of earning more than 50k?

<p style="color:#1A5276;"></b>Variables:</b><br>
Gender → categorical (Male, Female)
Income → categorical (≤50K, >50K)
Because both variables are categories, the Chi‑Square test is the right choice.
<p style="color:#1A5276;">
Null Hypothesis (H₀):
Gender and income category are independent.
(Gender does NOT affect whether someone earns more than $50k.)
    
<p style="color:#1A5276;">
Alternative Hypothesis (H₁):
Gender and income category are not independent.
(Gender DOES affect the likelihood of earning more than $50k.)
</p>


In [23]:
import scipy.stats as stats


# Create contingency table
table = pd.crosstab(df_train_test['sex'], df_train_test['income'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)
print("\nExpected Frequencies:\n", expected)

Chi-square: 2248.847679013691
Degrees of freedom: 1
p-value: 0.0

Expected Frequencies:
 [[12317.54964989  3874.45035011]
 [24837.45035011  7812.54964989]]


In [24]:
alpha = 0.05

if p < alpha:
    print("Reject H0: Gender and income are significantly related.")
else:
    print("Fail to reject H0: No significant relationship between gender and income.")

Reject H0: Gender and income are significantly related.


<p style="color:#2C3E50;">
<b>Based on the chi‑square test, the p‑value is much smaller than 0.05. This means the result is statistically significant. So, we reject the null hypothesis and conclude that gender and income level (earning more than 50K or not) are related in this dataset.
In simple terms,Gender appears to have an influence on whether a person earns more than 50K in the Adult Income dataset.
This does not explain why the difference exists, but it shows that the pattern is not due to random chance.
</p>

<h3 style="color:#1A5276;">Hypothesis Question 2:Does the average hours worked per week affect the probability of earning >50k?

<p style="color:#1A5276;"> Variables:
Hours per week → numerical
Income category → categorical (≤50K or >50K)
<p style="color:#1A5276;">
Null Hypothesis (H₀):
There is no difference in average hours worked per week between people earning ≤50K and >50K
<p style="color:#1A5276;">
Alternative Hypothesis (H₁):
People who earn >50K work more hours per week on average.
</p>

In [27]:
import pandas as pd
from scipy.stats import ttest_ind

# Split into two income groups
low_income = df_train_test[df_train_test['income'] == '<=50K']['hours_per_week']
high_income = df_train_test[df_train_test['income'] == '>50K']['hours_per_week']

t_stat, p_value = ttest_ind(low_income, high_income, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -54.662247230930255
P-value: 0.0


<p style="color:#2C3E50;">
<b>The t‑test shows a very large difference in the average number of hours worked per week between people who earn more than 50K and those who earn 50K or less. The p‑value is effectively 0, which is far below the usual significance level of 0.05. Because of this, we reject the null hypothesis. This means that income classification is significantly related to the number of hours a person works per week. In simple terms, people who earn more than 50K tend to work more hours per week than those who earn 50K or less.
</p>

<h3 style="color:#1A5276;">Hypothesis Question 3:Does Gender affect the income classification when the education is the same?

<p style="color:#1A5276;">
Null Hypothesis (H₀):
Gender does not affect income classification when education level is the same.

<p style="color:#1A5276;">
Alternative Hypothesis (H₁):
Males have a higher probability of earning more than $50,000 than females at the same education level.

</p>

In [29]:
import pandas as pd
from scipy.stats import chi2_contingency

# Get all unique education levels 
education_levels = df_train_test['education'].unique()

for edu in education_levels: 
    print(f"\nEducation Level: {edu}") # Filter data for this education level 
    subset = df_train_test[df_train_test['education'] == edu]


Education Level: Bachelors

Education Level: HS-grad

Education Level: 11th

Education Level: Masters

Education Level: 9th

Education Level: Some-college

Education Level: Assoc-acdm

Education Level: Assoc-voc

Education Level: 7th-8th

Education Level: Doctorate

Education Level: Prof-school

Education Level: 5th-6th

Education Level: 10th

Education Level: 1st-4th

Education Level: Preschool

Education Level: 12th


In [32]:
# Create contingency table: gender vs income
table = pd.crosstab(subset['sex'], subset['income'])
print("Contingency Table:") 
print(table)

Contingency Table:
income  <=50K  >50K
sex                
Female    207     4
Male      402    44


In [33]:
 # Run chi-square test
chi2, p, dof, expected = chi2_contingency(table)
    
print(f"Chi-square: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p:.6f}")
    

Chi-square: 12.2831
Degrees of freedom: 1
P-value: 0.000457


In [35]:
 # Interpretation
alpha = 0.05
if p < alpha:
    print("Conclusion: Reject H0 → Gender and income are significantly related at this education level.")
else:
    print("Conclusion: Fail to reject H0 → No significant relationship between gender and income at this education level.")

Conclusion: Reject H0 → Gender and income are significantly related at this education level.


<p style="color:#2C3E50;">
<b>The chi‑square test shows a statistically significant relationship between gender and income for this education level. Because the p‑value is much smaller than 0.05, we reject the null hypothesis. This means that even when people have the same level of education, gender still plays a role in whether they earn more than 50K in this dataset. In other words, income outcomes differ between men and women even within the same education group.
</p>

<h3 style="color:#1A5276;">Hypothesis Question 4:Does the age groups who worked more hours per week has income classification?

<p style="color:#1A5276;">
Null Hypothesis(H₀):
There is no significant difference in income classification between age groups.
<p style="color:#1A5276;">    
Alternative Hypothesis (H₁): 
Income classification differs significantly between age groups.
</p>

In [45]:
import pandas as pd
import statsmodels.api as sm
import patsy

# Create age groups
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ['18–25', '26–35', '36–45', '46–55', '56–65', '65+']
df_train_test['age_group'] = pd.cut(df_train_test['age'], bins=bins, labels=labels, right=False)

# Convert income to binary
df_train_test['income_binary'] = (df_train_test['income'] == '>50K').astype(int)

# Build logistic regression model using patsy
y, X = patsy.dmatrices('income_binary ~ age_group + hours_per_week', df_train_test, return_type='dataframe')


# Fit model
model = sm.Logit(y, X).fit()

print(model.summary())


Optimization terminated successfully.
         Current function value: 0.475799
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:          income_binary   No. Observations:                48842
Model:                          Logit   Df Residuals:                    48835
Method:                           MLE   Df Model:                            6
Date:                Mon, 09 Feb 2026   Pseudo R-squ.:                  0.1353
Time:                        14:06:22   Log-Likelihood:                -23239.
converged:                       True   LL-Null:                       -26875.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -5.7927      0.112    -51.852      0.000      -6.012      -5.574
age_g

Odds ratios gives how many times more likely each group is to earn >50K compared to the reference group

In [46]:
import numpy as np

# Odds ratios
odds_ratios = np.exp(model.params)
print(odds_ratios)


Intercept              0.003050
age_group[T.26–35]    13.678938
age_group[T.36–45]    32.913432
age_group[T.46–55]    43.579147
age_group[T.56–65]    34.381239
age_group[T.65+]      26.073179
hours_per_week         1.037021
dtype: float64


Income increases dramatically with age.
The likelihood of earning >50K rises sharply from early adulthood, peaks around 46–55, and then declines slightly but remains high.

Odds Ratio = 1.037,. Each additional hour worked per week increases the odds of earning >50K by about 3.7%.
Even small increases in weekly hours have a measurable impact on income classification.

<p style="color:#2C3E50;">
<b>The odds ratios show that both age group and hours worked per week have strong and statistically significant effects on income classification. Compared to individuals aged 18–25, all older age groups have substantially higher odds of earning more than 50K, with the highest likelihood observed in the 46–55 age range. Additionally, each extra hour worked per week increases the odds of earning >50K by approximately 3.7%. These results provide clear evidence that income classification differs significantly across age groups and is also influenced by the number of hours worked.
</b>